In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
# In Colab the package AND data/ must be present, so we clone the repo (Colab opens
# only the single .ipynb, not the repo). In CI/local the package is already installed
# from the checkout under test, so we skip the clone and exercise that code.
import os

try:
    import pocketlm  # already installed (local/CI) -> use the code under test
except ImportError:
    import subprocess
    import sys

    if not os.path.isdir("pocketlm-repo"):
        subprocess.check_call(
            ["git", "clone", "--depth", "1",
             "https://github.com/ni5h4nt/pocketlm", "pocketlm-repo"])
    os.chdir("pocketlm-repo")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
    import pocketlm  # noqa: F811

import torch

# nbmake runs a notebook from its own directory; anchor the cwd at the repo root
# (derived from the installed package) so CORPUS_PATH resolves in CI, locally, and
# in Colab (where the except-branch already chdir'd into the clone).
os.chdir(os.path.dirname(os.path.dirname(os.path.abspath(pocketlm.__file__))))

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = "data/corpus.txt"   # repo-root-relative; valid after the chdir above
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l7.5 Preference optimization

SFT teaches *a* good answer; preference methods teach *which of two* answers humans
prefer. **DPO** does this directly on `(chosen, rejected)` pairs against a frozen
reference model, no separate reward model or RL loop.

In [ ]:
from pocketlm import dpo_loss

ref_chosen, ref_rejected = torch.tensor(-2.0), torch.tensor(-2.0)
good = dpo_loss(torch.tensor(-1.0), torch.tensor(-3.0), ref_chosen, ref_rejected)
bad = dpo_loss(torch.tensor(-3.0), torch.tensor(-1.0), ref_chosen, ref_rejected)
print("loss when policy prefers the chosen reply: ", round(good.item(), 3))
print("loss when policy prefers the rejected reply:", round(bad.item(), 3))

DPO's loss is low exactly when the policy raises the chosen-over-rejected margin
beyond the reference. Optimizing it nudges the model toward human-preferred
responses, the alignment step after SFT.

In [ ]:
assert good < bad